# Toy Environment
# 0. 介绍
**研究背景**：大模型只能给出下一步动作，真正执行动作、保存状态和返回结果的是外部环境。不同任务可能使用文件、网页或终端等不同环境，因此 Agent 需要一套统一的接口与它们交互。

**现存问题**：不同环境的操作名称、返回格式和成功标准可能完全不同。如果 Agent 的执行代码直接依赖某一种环境，更换环境后，即使大模型给出了正确动作，程序也可能无法执行；环境不能可靠重置时，上一次运行留下的状态还会干扰下一次测试。

**解决方案**：本 Notebook 将实现一个极简的 Toy Environment，用 `reset()`、`step()`、`snapshot()` 和 `success()` 统一环境的重置、执行、观察和验收。然后用同一份真实 API 动作进行对比：基线版本直接依赖具体环境而失败，改进版本通过统一接口正确执行，从而直观看到环境合同如何让 Agent 稳定地操作不同环境。
## 目录
0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型要完成的任务，以及程序接收和处理结果的方式。

# 2. 前置准备
## 2.1 说明可用工具
大模型需要先知道自己可以执行什么动作。本节提供一个 `set_counter` 工具，告诉大模型可以把计数器设置为指定整数。

In [2]:
tools = [{
    "type": "function",
    "function": {
        "name": "set_counter",
        "description": "把计数器设置为指定整数",
        "parameters": {
            "type": "object",
            "properties": {"value": {"type": "integer"}},
            "required": ["value"],
        }}}]
print(f"可用工具：{tools[0]['function']['name']}")

可用工具：set_counter


输出显示 `set_counter` 已经准备好，说明大模型知道了工具名称和需要填写的整数。下一步会给大模型一个明确的计数器目标。

## 2.2 写出具体任务
有了工具说明，还需要告诉大模型这次要完成什么。本节要求大模型调用刚才的工具，把计数器设置为 `3`。

In [3]:
messages = [
    {"role": "system", "content": "你是计数器助手，只能调用 set_counter。"},
    {"role": "user", "content": "请把计数器设置为 3。"},
]
print(f"任务：{messages[-1]['content']}")

任务：请把计数器设置为 3。


输出显示了大模型将要完成的目标。下一步会创建一个最简单的外部环境，用来保存计数器的真实状态。

## 2.3 创建环境状态
模型给出的动作不会自动改变外部世界。本节用一个字典代表最小环境，其中 `counter` 保存计数器当前的真实数值。

In [4]:
environment_state = {"counter": 0}
print(f"环境初始状态：{environment_state}")

环境初始状态：{'counter': 0}


输出中的 `counter` 仍为 `0`，说明此时只是创建了环境，大模型尚未给出动作，程序也尚未执行动作。下一步会规定任务完成的唯一标准。

## 2.4 定义成功标准
不能只听大模型说任务已经完成，程序需要检查环境的真实状态。本节规定只有 `counter` 等于目标值 `3`，任务才算成功。

In [5]:
target_value = 3

def grade(state):
    return state["counter"] == target_value

print(f"成功标准：counter == {target_value}")

成功标准：counter == 3


输出显示了唯一的成功标准，说明后面的基线版本和改进版本会用同一把尺子判断结果。至此，工具、任务、环境状态和成功标准都已准备完成，下一章将调用真实大模型并查看它返回的动作。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
模型、工具和任务已经准备好，现在可以把它们一起发给大模型。本节要求大模型必须选择工具，并记录从发出请求到收到回复所用的时间。

In [6]:
import json
from time import perf_counter

start = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",  # 必须选择一个工具
    temperature=0,
)
latency_ms = round((perf_counter() - start) * 1000)
raw_real = response.model_dump()
print(f"回复已收到：provider={config['NANO_BACKEND']}, model={model_name}, latency={latency_ms} ms")
print("具体内容如下：")
print(json.dumps(raw_real, indent=4, ensure_ascii=False))

回复已收到：provider=openai, model=LongCat-2.0, latency=2163 ms
具体内容如下：
{
    "id": "e1746036f8e0442ca5f14b0047361bb7",
    "choices": [
        {
            "finish_reason": "tool_calls",
            "index": 0,
            "logprobs": null,
            "message": {
                "content": null,
                "refusal": null,
                "role": "assistant",
                "annotations": null,
                "audio": null,
                "function_call": null,
                "tool_calls": [
                    {
                        "id": "call_d31819e1a7c44b4b91356c28",
                        "function": {
                            "arguments": "{\"value\": 3}",
                            "name": "set_counter"
                        },
                        "type": "function",
                        "index": null
                    }
                ],
                "reasoning_content": "\n用户要求将计数器设置为3。我需要使用set_counter函数，参数value设置为3。"
            },
            

输出显示了模型来源、模型名称和等待时间，说明真实大模型已经返回结果，完整内容保存在 `raw_real` 中。下一步会读取这份结果，查看大模型给出的具体动作。

## 3.2 查看并保存动作
大模型已经返回结果，但外部环境需要一个简单明确的动作才能执行。本节读取工具名称和参数并保存为 `action`，同时显示停止原因和 Token 用量。

In [7]:
choice = raw_real["choices"][0]
tool_call = choice["message"]["tool_calls"][0]
arguments = json.loads(tool_call["function"]["arguments"])
action = {"name": tool_call["function"]["name"], "value": arguments["value"]}
print(f"动作：{action}")
print(f"停止原因：{choice['finish_reason']}")
print(f"Token 用量：{raw_real['usage']}")

动作：{'name': 'set_counter', 'value': 3}
停止原因：tool_calls
Token 用量：{'completion_tokens': 37, 'prompt_tokens': 153, 'total_tokens': 190, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 20, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': 0, 'video_tokens': 0, 'text_tokens': 0}, 'effectiveCachedTokens': 128, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'output_tokens_details': None, 'cached_tokens': 0}


输出显示大模型选择了 `set_counter` 并给出目标值 `3`，说明模型本身已经做出了正确决定。`action` 保存了后续实验共用的真实动作；停止原因、Token 用量和延迟记录了这次请求的运行情况，下一章将定义直接依赖具体环境的基线组件。

# 4. 定义基线组件
## 4.1 定义直接执行器
最简单的做法是让执行代码直接修改环境字段，但它必须提前知道字段名称。本节先保留这种做法作为基线：执行器假定环境使用 `value` 保存计数器，并把模型给出的数值直接写入该字段。

In [8]:
class DirectRunner:
    def run(self, state, action):
        state["value"] = action["value"]  # 写死具体环境的字段名
        return state

baseline = DirectRunner()
print("基线组件：直接写入 value 字段")

基线组件：直接写入 value 字段


输出说明基线组件已经定义完成，但尚未修改环境。它只认识自己写死的 `value` 字段，不知道当前环境使用的是 `counter`；下一章将让它执行第 3 章保存的真实动作，查看环境状态是否正确改变。

# 5. 展示基线故障
## 5.1 执行真实动作
第 3 章已经保存了大模型给出的正确动作，现在把它交给基线执行器。本节保留动作前后的环境状态，直接查看计数器是否被写到了正确位置。

In [9]:
baseline_before = environment_state.copy()
baseline_after = baseline.run(environment_state.copy(), action)
print(f"真实动作：{action}")
print(f"执行前：{baseline_before}")
print(f"执行后：{baseline_after}")

真实动作：{'name': 'set_counter', 'value': 3}
执行前：{'counter': 0}
执行后：{'counter': 0, 'value': 3}


输出显示模型要求把数值设为 `3`，但环境中的 `counter` 仍为 `0`，同时多出了基线写入的 `value`。这说明模型动作没有错，直接执行器却因字段名称不同而改错了位置；下一步会用统一标准判断任务结果。

## 5.2 判断任务结果
环境状态已经展示了问题，还需要用第 2 章定义的同一成功标准给出明确结论。本节检查真正的 `counter` 是否达到目标值，并保存基线结果。

In [10]:
baseline_passed = grade(baseline_after)
print(f"基线任务通过：{baseline_passed}")

基线任务通过：False


输出为 `False`，说明基线做法没有完成任务。真实大模型给出了正确动作，但绑定具体字段的执行代码无法适应当前环境；下一章将定义统一环境接口，让执行代码不再直接修改内部状态。

# 6. 定义改进组件
## 6.1 定义统一环境接口
基线失败是因为执行器直接修改了环境内部字段。改进方法是把内部状态藏在 `ToyEnv` 后面：`reset()` 重置环境，`step()` 执行动作，`snapshot()` 返回当前状态，`success()` 判断任务是否完成。

In [11]:
class ToyEnv:
    def __init__(self, target):
        self.target = target
        self.reset()

    def reset(self):
        self.state = {"counter": 0}
        return self.snapshot()

    def step(self, action):
        self.state["counter"] = action["value"]  # 在环境内部完成字段转换
        return self.snapshot()

    def snapshot(self):
        return self.state.copy()

    def success(self):
        return self.state["counter"] == self.target

fixed = ToyEnv(target_value)
print("改进组件：reset、step、snapshot、success")

改进组件：reset、step、snapshot、success


输出说明统一环境接口已经定义完成。外部执行代码只需调用四个固定方法，不再读取或修改 `counter`；下一章将让它执行与基线完全相同的真实动作，查看任务能否完成。

# 7. 展示修复结果
## 7.1 重置环境
为了让每次实验都从相同状态开始，本节先调用 `reset()` 清除旧状态。它会同时返回一份状态快照，方便直接查看新的起点。

In [12]:
fixed_before = fixed.reset()
print(f"重置后的状态：{fixed_before}")

重置后的状态：{'counter': 0}


输出中的 `counter` 为 `0`，说明环境已经回到固定的初始状态，不会受到之前运行的影响。下一步会通过统一的 `step()` 执行与基线完全相同的真实动作。

## 7.2 执行真实动作
本节把第 3 章保存的同一个 `action` 交给 `step()`。执行代码只传入动作，不需要知道环境内部使用哪个字段，`step()` 会完成转换并返回新的状态快照。

In [13]:
fixed_after = fixed.step(action)
print(f"真实动作：{action}")
print(f"执行后状态：{fixed_after}")

真实动作：{'name': 'set_counter', 'value': 3}
执行后状态：{'counter': 3}


输出显示 `counter` 已从 `0` 变为 `3`，说明真实模型动作被写到了环境的正确位置。下一步会让环境根据自己的真实状态判断任务是否完成。

## 7.3 判断任务结果
状态已经正确改变，还需要得到明确的任务结论。本节调用 `success()`，由环境检查 `counter` 是否达到第 2 章规定的同一目标值。

In [14]:
fixed_passed = fixed.success()
print(f"改进任务通过：{fixed_passed}")

改进任务通过：True


输出为 `True`，说明统一环境接口成功完成了任务。基线和改进版本使用同一个真实模型动作与目标值，结果差异只来自环境交互方式；下一章将汇总完整对照。

# 8. 汇总消融对照
## 8.1 对比两种做法
只改变环境交互方式并比较前后结果，就能看出统一接口是否重要。本节先汇总两种做法共同使用的真实 API 信息，再并排记录执行后的环境状态和任务结果。

In [15]:
shared_info = {
    "模型来源": config["NANO_BACKEND"],
    "模型": model_name,
    "真实动作": action,
    "等待时间（毫秒）": latency_ms,
    "Token 总量": raw_real["usage"]["total_tokens"],
    "停止原因": choice["finish_reason"],
}
comparison = [
    {"做法": "直接修改字段", "执行后状态": baseline_after, "任务通过": baseline_passed},
    {"做法": "ToyEnv 统一接口", "执行后状态": fixed_after, "任务通过": fixed_passed},
]
print("共同信息：")
print(json.dumps(shared_info, ensure_ascii=False, indent=2))
print("消融对照：")
print(json.dumps(comparison, ensure_ascii=False, indent=2))

共同信息：
{
  "模型来源": "openai",
  "模型": "LongCat-2.0",
  "真实动作": {
    "name": "set_counter",
    "value": 3
  },
  "等待时间（毫秒）": 2163,
  "Token 总量": 190,
  "停止原因": "tool_calls"
}
消融对照：
[
  {
    "做法": "直接修改字段",
    "执行后状态": {
      "counter": 0,
      "value": 3
    },
    "任务通过": false
  },
  {
    "做法": "ToyEnv 统一接口",
    "执行后状态": {
      "counter": 3
    },
    "任务通过": true
  }
]


输出显示两种做法使用同一份真实模型动作：直接修改字段时，真正的 `counter` 没有改变，任务失败；通过 ToyEnv 统一接口时，`counter` 正确变为 `3`，任务通过。模型没有改变，决定结果的是模型外层的环境合同，至此本 Notebook 的对照实验结束。

## 8.2 拓展

### nano 版省略了什么

nano 版环境只有内存状态和同步 step，没有覆盖真实文件系统、浏览器或终端、随机种子、快照、并发动作、部分可观测性、超时、外部副作用和环境版本。生产评测环境还需可复位、可重放并提供独立 grader；但核心契约仍是动作必须通过环境接口改变可观测状态。

### 延伸阅读


1. 2024, [SWE-agent: Agent-Computer Interfaces Enable Automated Software Engineering](https://arxiv.org/abs/2405.15793)：Agent-Computer Interface 如何塑造环境动作与反馈。
2. 2025, [OpenHands: An Open Platform for AI Software Developers](https://openreview.net/forum?id=OJd3ayDDoF)：真实软件环境中的通用 Agent 平台与可复现实验。
3. 2024, [tau-bench: A Benchmark for Tool-Agent-User Interaction](https://arxiv.org/abs/2406.12045)：带状态规则和用户交互的环境级任务判定。